# Oferta Hídrica

> **Contexto de dominio:** [`docs/fuentes/oferta_hidrica.md`](../../docs/fuentes/oferta_hidrica.md)  
> **Bloque:** A | **Línea:** `oferta_hidrica`  
> **Variable principal:** `caudal` (m³/s)  
> **Modelos sugeridos:** SARIMA, SARIMAX, PROPHET, XGBOOST, RANDOM_FOREST  
> Flujo: `Plan.md` sección 3 — ciclo estadístico completo.

**Antes de comenzar:** Leer `docs/fuentes/oferta_hidrica.md` para entender:
- Variables ambientales clave y sus rangos físicos
- Normativa colombiana aplicable (umbrales normativos)
- Fuentes de datos oficiales y frecuencia de actualización
- Preguntas analíticas típicas de esta línea

## 0. Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from estadistica_ambiental.io.loaders import load_csv
from estadistica_ambiental.io.validators import validate
from estadistica_ambiental.eda.variables import classify
from estadistica_ambiental.eda.quality import assess_quality
from estadistica_ambiental.eda.profiling import run_eda
from estadistica_ambiental.eda.viz import plot_series, plot_seasonal_means
from estadistica_ambiental.preprocessing.imputation import impute
from estadistica_ambiental.descriptive.univariate import summarize
from estadistica_ambiental.inference.stationarity import stationarity_report
from estadistica_ambiental.inference.trend import mann_kendall
from estadistica_ambiental.inference.intervals import exceedance_report
from estadistica_ambiental.features.climate import load_oni, enso_lagged
from estadistica_ambiental.config import ENSO_LAG_MESES
from estadistica_ambiental.predictive.registry import get_model, list_models
from estadistica_ambiental.evaluation.backtesting import walk_forward
from estadistica_ambiental.evaluation.comparison import rank_models
from estadistica_ambiental.config import DOCS_FUENTES

LINEA = "oferta_hidrica"
VARIABLE = "caudal"
UNIDAD = "m³/s"

print("Setup OK | Modelos disponibles:", list_models())

## 0b. Contexto de dominio
> Carga la ficha técnica de la línea `oferta_hidrica` para tener presente la normativa, los indicadores y las preguntas analíticas durante el análisis.

In [ ]:
ficha = DOCS_FUENTES / "oferta_hidrica.md"
if ficha.exists():
    print(ficha.read_text(encoding="utf-8")[:3000])  # primeras 3000 chars
else:
    print("Ficha no encontrada en", ficha)

## 1. Cargar datos
> Colocar el archivo en `data/raw/` y ajustar la ruta.  
> Ver sección **Datos y fuentes** de `docs/fuentes/oferta_hidrica.md` para las fuentes oficiales.

In [ ]:
# df = load_csv("data/raw/oferta_hidrica.csv", date_col="fecha")

# --- Datos sintéticos de ejemplo ---
np.random.seed(42)
n = 120
df = pd.DataFrame({
    "fecha":    pd.date_range("2015-01-01", periods=n, freq="ME"),
    "caudal": np.random.gamma(3, 5, n) + np.linspace(0, 5, n),
})
print(f"Shape: {df.shape} | Rango: {df.fecha.min()} → {df.fecha.max()}")
df.head()

<!-- ENRICHMENT: oferta_hidrica -->

## 1b. Hidrogeologia: Piezometria, transmisividad y curva de abatimiento

El inventario FUNIAS levanta los parametros clave del acuifero por pozo:

| Parametro | Simbolo | Unidad | Rango tipico |
|---|---|---|---|
| Nivel estatico | Ne | m | 3 - 30 m |
| Nivel dinamico | Nd | m | > Ne |
| Transmisividad | T | m2/dia | 10 - 5000 m2/dia |
| Caudal extraccion | Q | L/s | 1 - 50+ L/s |

**Ecuacion de Theis** (bombeo a largo plazo, sin recarga):
```
s(r,t) = Q/(4*pi*T) * W(u)       u = r^2*S/(4*T*t)
```
Donde W(u) = funcion de pozo (well function), r = distancia al pozo (m), S = coeficiente de almacenamiento (adim), t = tiempo (dias).

**Kriging Empirico Bayesiano (EBK):** mejor opcion para mapas de isopiezas porque estima la varianza del modelo variografico en lugar de asumirla fija.

**MODFLOW** (USGS): modelo de diferencias finitas para simular el acuifero en 3D; se conecta a Python mediante `flopy`.

In [ ]:
# Inventario FUNIAS sintetico + curva de abatimiento (Theis) + Kriging
np.random.seed(33)
n_pozos = 25

# Inventario FUNIAS (pozos del acuifero)
df_pozos = pd.DataFrame({
    'pozo_id': [f'PZ-{i+1:03d}' for i in range(n_pozos)],
    'ne_m': np.random.uniform(3, 25, n_pozos).round(1),        # nivel estatico
    'q_ls': np.random.uniform(1, 40, n_pozos).round(1),         # caudal extraccion
    'transmisiv': np.random.lognormal(4, 1, n_pozos).round(1),  # m2/dia
    'lat': np.random.uniform(4.5, 6.5, n_pozos),                # lat Colombia
    'lon': np.random.uniform(-75.5, -73.5, n_pozos),            # lon Colombia
})
df_pozos['nivel_piezom'] = df_pozos['ne_m'] + np.random.uniform(50, 200, n_pozos)  # cota

# Curva de abatimiento Theis para pozo representativo
Q_bomba = 10.0    # L/s = 0.01 m3/s
T_trans = 150.0   # m2/dia (transmisividad tipica)
S_almac = 0.001   # coeficiente almacenamiento (acuifero confinado)
r_obs = 50.0      # m distancia pozo de observacion
t_dias = np.logspace(-2, 2, 200)  # 0.01 a 100 dias

# Aproximacion Cooper-Jacob (valida para u < 0.05)
u = r_obs**2 * S_almac / (4 * T_trans * t_dias)
abatim = np.where(u < 0.05,
    (Q_bomba * 86.4) / (4 * np.pi * T_trans) * (-0.5772 - np.log(u)),
    np.nan)

print(f'Inventario FUNIAS: {n_pozos} pozos')
print(f'Transmisividad | mediana={df_pozos["transmisiv"].median():.1f} m2/dia')
print(f'Caudal medio extraccion: {df_pozos["q_ls"].mean():.1f} L/s')
print(f'Abatimiento maximo en r={r_obs}m despues de 100 dias: {np.nanmax(abatim):.2f} m')
print()
print('Para mapa de isopiezas: usar pykrige.OrdinaryKriging o EmpiricalBayesianKriging')
print('Para modelo MODFLOW 3D: flopy.modflow (USGS) con Python/FloPy')
df_pozos[['pozo_id','ne_m','q_ls','transmisiv','nivel_piezom']].head(10)

## 2. Validación y EDA
> `validate()` usa rangos físicos específicos para `oferta_hidrica` desde `config.py`.

In [ ]:
val = validate(df, date_col="fecha", linea_tematica=LINEA)
print(val.summary())

In [ ]:
run_eda(df, output=f"data/output/reports/eda_oferta_hidrica.html",
       title="EDA — Oferta Hídrica", date_col="fecha", use_ydata=False)
# Abrir el HTML en el navegador para el reporte completo

## 3. Visualización exploratoria

In [ ]:
plot_series(df, "fecha", "caudal", title="Oferta Hídrica — caudal (m³/s)")
plt.show()

In [ ]:
plot_seasonal_means(df, "fecha", "caudal", period="month")
plt.show()

## 3c. Mapa de isopiezas y curva de abatimiento — Kriging + Theis

El mapa de isopiezas (lineas de igual carga hidraulica) muestra la direccion del flujo subterraneo y permite identificar zonas de recarga (valores altos) y descarga (valores bajos). Se genera con **Kriging Ordinario** o **EBK** sobre los niveles piezometricos de los pozos FUNIAS.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: Dispersion de pozos FUNIAS + nivel piezometrico (proxy mapa isopiezas)
sc = ax1.scatter(df_pozos['lon'], df_pozos['lat'],
                 c=df_pozos['nivel_piezom'], cmap='Blues_r', s=60, alpha=0.85, edgecolors='gray')
plt.colorbar(sc, ax=ax1, label='Nivel piezometrico (m.s.n.m.)')
ax1.set_xlabel('Longitud'); ax1.set_ylabel('Latitud')
ax1.set_title('Pozos FUNIAS — Mapa de isopiezas\n(Kriging EBK con pykrige)', fontweight='bold')
ax1.grid(alpha=0.3)
# Anotar pozos con mayor transmisividad
top3 = df_pozos.nlargest(3, 'transmisiv')
for _, row in top3.iterrows():
    ax1.annotate(f"{row['pozo_id']}\nT={row['transmisiv']:.0f}",
                 (row['lon'], row['lat']), fontsize=6, color='darkblue')

# Panel B: Curva de abatimiento Theis / Cooper-Jacob
ax2.semilogx(t_dias, abatim, lw=2, color='#e74c3c', label=f'Abatimiento r={r_obs}m')
ax2.axhline(5.0, color='orange', ls='--', lw=1.5, label='Umbral abatim. 5m (alerta)')
ax2.axhline(10.0, color='red', ls='--', lw=1.5, label='Umbral abatim. 10m (critico)')
ax2.set_xlabel('Tiempo de bombeo (dias, escala log)')
ax2.set_ylabel('Abatimiento (m)')
ax2.set_title(f'Curva abatimiento Theis/Cooper-Jacob\nQ={Q_bomba} L/s, T={T_trans} m2/dia', fontweight='bold')
ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

plt.suptitle('Hidrogeologia — FUNIAS + Theis + Kriging (MODFLOW/FloPy para modelo 3D)',
             fontweight='bold', fontsize=11)
plt.tight_layout(); plt.show()

t_alerta = t_dias[np.nanargmax(abatim >= 5.0)] if (abatim >= 5.0).any() else None
if t_alerta:
    print(f'Abatimiento supera 5m despues de {t_alerta:.1f} dias de bombeo')

## 3b. Covariable ENSO (ONI)
> Lag recomendado para `oferta_hidrica` definido en `config.ENSO_LAG_MESES`.

In [ ]:
# --- Covariable ENSO (lag específico para Oferta Hídrica) ---
# Guard (M-20): avisar si LINEA no tiene lag específico — se aplicará el default.
if LINEA not in ENSO_LAG_MESES:
    _lag_default = ENSO_LAG_MESES["default"]
    print(f"[aviso] '{LINEA}' sin lag específico en ENSO_LAG_MESES; "
          f"se usará default ({_lag_default} meses).")
else:
    print(f"[ok] ENSO lag para '{LINEA}' = {ENSO_LAG_MESES[LINEA]} meses")

oni = load_oni()  # Descarga ONI desde NOAA
df = enso_lagged(df, oni, date_col="fecha", linea_tematica=LINEA)
print("Columnas ENSO agregadas:", [c for c in df.columns if "oni" in c or "enso" in c])

## 4. Estadística descriptiva

In [ ]:
summarize(df)

## 5. Inferencial
> ADR-004: Estacionariedad obligatoria pre-ARIMA (ADF + KPSS juntos).

In [ ]:
ts = df.set_index("fecha")["caudal"].dropna()
stationarity_report(ts)

In [ ]:
mk = mann_kendall(ts)
print(f"Tendencia: {mk['trend']} | p={mk['pval']:.4f} | slope={mk['slope']:.6f}")

## 5b. Análisis de excedencias normativas
> Compara `caudal` contra las normas colombianas relevantes.

In [ ]:
rep = exceedance_report(df["caudal"], variable="caudal")
if rep.empty:
    print("Sin normas colombianas registradas para 'caudal'. "
          "Agregar umbral manual a la llamada exceedance_probability().")
else:
    display(rep)

## 6. Preprocesamiento

In [ ]:
df_clean = impute(df.copy(), cols=["caudal"], method="linear")
print(f"Faltantes antes: {df["caudal"].isna().sum()} | "
      f"después: {df_clean["caudal"].isna().sum()}")

## 7. Modelos predictivos

In [ ]:
ts = df_clean.set_index("fecha")["caudal"]

models = {
    "SARIMA":       get_model("sarima", order=(1,1,1), seasonal_order=(1,1,1,12)),
    "SARIMAX":      get_model("sarimax", order=(1,1,1), seasonal_order=(1,1,1,12)),
    "Prophet":      get_model("prophet"),
    "XGBoost":      get_model("xgboost", lags=[1,2,3,6,12]),
    "RandomForest": get_model("random_forest", lags=[1,2,3,6,12]),
}

results = {}
for name, model in models.items():
    if name.startswith("#"):
        continue
    results[name] = walk_forward(model, ts, gap=12, horizon=6, n_splits=4)
    print(f"{name}: RMSE={results[name]['metrics']['rmse']:.3f}")

In [ ]:
rank_models(results)[["rmse","mae","r2","score","rank"]]

## 7b. Guardrails y supuestos metodológicos
<!-- GUARDRAILS: oferta_hidrica -->

> **Antes de publicar resultados**, verificar que se cumplen los supuestos clave del flujo. Esta sección lista los más comunes y los específicos de la línea.

### Supuestos comunes (todas las líneas)

- **Estacionariedad (ADR-004):** ADF + KPSS deben coincidir antes de aplicar ARIMA. Si discrepan, diferenciar conservadoramente o usar modelos no-ARIMA (Prophet, ML).
- **Outliers (ADR-002):** los picos ambientales son señal real (eventos, episodios). No aplicar clipping automático — sólo `preprocessing/outliers.py` opt-in y documentado.
- **Métrica primaria (ADR-003):** RMSLE NO en variables que pueden ser negativas o cero. Usar MAE + sMAPE (o NSE / KGE en hidrología) como default.
- **Tamaño muestral mínimo:** ARIMA requiere ≥ 36 observaciones; STL anual con datos diarios, ≥ 2 ciclos completos.
- **Residuos (post-fit):** verificar normalidad (Jarque-Bera) e independencia (Ljung-Box, lag = 12). Residuos correlacionados → modelo subespecificado.
- **Walk-forward con gap (BP-1):** series con ACF ≥ 0.7 inflan R². Usar `gap ≥ horizonte` en `walk_forward()`.
- **Normas oficiales:** usar `config.NORMA_*` y `config.*_THRESHOLDS` — nunca umbrales hardcodeados en el notebook (ADR-005).

### Supuestos específicos — Hidrología

- **NSE / KGE primarias:** R² no es confiable en series con alta variabilidad estacional.
- **Lag ENSO = 4 meses** (cuencas Magdalena-Cauca, respuesta hidrológica rezagada).
- **Calibración / validación:** mínimo 70/30 split sobre años hidrológicos completos.
- **Caudales máximos / mínimos:** preservar — son la señal de eventos críticos (creciente, estiaje).

### Antes de presentar a la autoridad ambiental

- Reportar intervalos de confianza, no sólo el punto estimado.
- Documentar el período de los datos, los gaps y el método de imputación usado.
- Registrar decisiones metodológicas no triviales en `docs/decisiones.md` (ADRs).


## 8. Conclusiones

- **Línea temática:** Oferta Hídrica (`oferta_hidrica`)
- **Variable analizada:** `caudal` (m³/s)
- **Modelos ejecutados:** SARIMA, SARIMAX, PROPHET, XGBOOST, RANDOM_FOREST
- Completar con hallazgos reales al trabajar con datos de producción.

### Normativa y referencias
- Ver `docs/fuentes/oferta_hidrica.md` para normativa colombiana, indicadores oficiales y fuentes de datos.
- Registrar decisiones metodológicas en `docs/decisiones.md`.